# 05 · Classical baselines (plan steps 3.1–3.2)

The assignment asks for a comparison with a **simple baseline**. This notebook:

1. describes every train and val image with hand-crafted features at 128 × 128: HSV colour histogram (96),
   HOG (1,764) and uniform LBP histogram (10) → 1,870 values (`plantdx.baselines.features`);
2. trains **logistic regression** and a **linear SVM** (`class_weight="balanced"`), with HOG reduced to 256
   PCA components (D-011), for C ∈ {0.01, 0.1, 1, 10};
3. picks C on **val macro-F1** and writes one row per model to the shared results table.

The test split is not touched (it is scored once, in step 4.4). **No GPU needed** (a CPU runtime is fine).
Measured on a Colab CPU runtime: features about 11 minutes the first time (cached on Drive afterwards); the
C grid about 40 minutes, most of it the linear SVM at C = 10, which does not converge (D-011).

## 1 · Setup

In [ ]:
import os, sys
from google.colab import drive

drive.mount("/content/drive")
REPO_URL = "https://github.com/AnarHummatli/plant-disease-detection.git"
REPO_DIR = "/content/plant-disease-detection"
if os.path.isdir(f"{REPO_DIR}/.git"):
    # the Colab copy is disposable: always match GitHub exactly (D-009)
    !git -C {REPO_DIR} fetch -q origin && git -C {REPO_DIR} reset -q --hard origin/main
else:
    !git clone -q {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
%pip install -q -e ".[dev]"
if f"{REPO_DIR}/src" not in sys.path:
    sys.path.insert(0, f"{REPO_DIR}/src")
os.environ["PLANTDX_ARTIFACTS"] = "/content/drive/MyDrive/plantdx/artifacts"
head = !git -C {REPO_DIR} rev-parse HEAD
remote = !git -C {REPO_DIR} rev-parse origin/main
assert head == remote, "repo is not at the latest commit: run this cell again"
!git -C {REPO_DIR} log -1 --oneline
!nvidia-smi -L || echo "no GPU"
!python -m plantdx.env

## 2 · Images

In [ ]:
from plantdx.data.images import download_archive, extract_variant

VARIANT, DATA_ROOT = "color", "/content/data"
print("new files extracted:", extract_variant(download_archive(), DATA_ROOT, VARIANT))

## 3 · Features + C grid

Each line is one C value: val macro-F1, val accuracy, fit time. `not converged` means the solver hit its
iteration limit; the score is still valid, but report it if it happens for the chosen C.

In [ ]:
from plantdx.baselines.run import run

table = run(DATA_ROOT)
cols = ["run_id", "model", "split", "n", "accuracy", "macro_f1", "weighted_f1", "top3", "size_mb", "train_seconds"]
table[cols]

## 4 · Grid details and the weakest classes

In [ ]:
import json
import pandas as pd
from plantdx.paths import run_dir

for run_id in ("baseline_logreg", "baseline_linsvm"):
    print(run_id)
    display(pd.read_csv(run_dir(run_id) / "grid.csv"))
    rep = json.loads((run_dir(run_id) / "metrics_val.json").read_text())
    worst = sorted(rep["per_class"], key=lambda r: r["f1"])[:5]
    display(pd.DataFrame(worst)[["class", "support", "precision", "recall", "f1"]])

## 5 · Download the results

Unzip `step3_baselines.zip` **at the repository root** on your computer: it adds `reports/results.csv` and
`reports/runs/baseline_*/`. Models (`model.joblib`) stay on Drive.

In [ ]:
from google.colab import files
from plantdx.results import export_for_repo

out = export_for_repo(["baseline_logreg", "baseline_linsvm"], "/content/step3_baselines.zip")
files.download(str(out))